# Customer Insight AI: Customer Segmentation using K-Means Clustering
**Task 2 - SkillCraft Technology Machine Learning Internship**

### Objective
Develop a K-Means clustering algorithm to segment customers of a retail store based on their purchase history (Annual Income and Spending Score) to derive actionable business insights and marketing strategies.

## 1. Import Libraries
We import standard packages for data manipulation, visualization, scaling, and clustering.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import joblib

# Setup plotting styles
sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Load Dataset
We load the Mall Customers Dataset.

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
print(f"Dataset successfully loaded. Shape: {df.shape}")
df.head()

## 3. Data Dictionary & Overview

### Data Dictionary
| Column | Type | Description |
|---|---|---|
| **CustomerID** | Numerical (Discrete) | Unique identifier for each customer |
| **Gender** | Categorical | Gender of the customer (Male / Female) |
| **Age** | Numerical (Discrete) | Age of the customer in years |
| **Annual Income (k$)** | Numerical (Continuous) | Annual income of the customer in thousands of dollars |
| **Spending Score (1-100)** | Numerical (Discrete) | Score assigned by the mall based on customer behavior and spending patterns |

In [ ]:
print("=== Dataset Info ===")
df.info()

print("\n=== Summary Statistics ===")
display(df.describe())

print("\n=== Missing Values ===")
print(df.isnull().sum())

## 4. Exploratory Data Analysis (EDA)
Let's visually explore distributions, outliers, gender counts, and correlations.

In [ ]:
# 1. Distributions of Numerical Columns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Age'], kde=True, ax=axes[0], color='#7C3AED')
axes[0].set_title('Distribution of Age', fontsize=12, fontweight='bold')

sns.histplot(df['Annual Income (k$)'], kde=True, ax=axes[1], color='#06B6D4')
axes[1].set_title('Distribution of Annual Income', fontsize=12, fontweight='bold')

sns.histplot(df['Spending Score (1-100)'], kde=True, ax=axes[2], color='#22C55E')
axes[2].set_title('Distribution of Spending Score', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Boxplots to Check for Outliers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(y=df['Age'], ax=axes[0], color='#7C3AED')
axes[0].set_title('Boxplot of Age (Outliers)', fontsize=12, fontweight='bold')

sns.boxplot(y=df['Annual Income (k$)'], ax=axes[1], color='#06B6D4')
axes[1].set_title('Boxplot of Annual Income (Outliers)', fontsize=12, fontweight='bold')

sns.boxplot(y=df['Spending Score (1-100)'], ax=axes[2], color='#22C55E')
axes[2].set_title('Boxplot of Spending Score (Outliers)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Gender Distribution Count Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='Gender', data=df, hue='Gender', palette=['#06B6D4', '#7C3AED'], legend=False)
plt.title('Gender Distribution Count', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(8, 6))
numerical_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
corr = df[numerical_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Pairplot colored by Gender
pairplot_fig = sns.pairplot(df.drop('CustomerID', axis=1), hue='Gender', palette=['#7C3AED', '#06B6D4'])
pairplot_fig.fig.suptitle('Pairplot of Features Colored by Gender', y=1.02, fontweight='bold')
plt.show()

## 5. Feature Selection & Feature Scaling

### Feature Selection
We select the primary columns for segmentation: `Annual Income (k$)` and `Spending Score (1-100)`.

### Why Scaling is Crucial for K-Means
K-Means is a distance-based clustering algorithm that uses **Euclidean distance** to calculate similarity between observations:

$$d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

Because `Annual Income (k$)` and `Spending Score (1-100)` have different numerical ranges and scales, scaling ensures both features contribute equally to the distance calculation, preventing one feature from dominating the objective function.

In [ ]:
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled features sample:")
print(X_scaled[:5])

## 6. Elbow Method & Silhouette Analysis
We run K-Means across a range of clusters $K \in [1, 10]$ to calculate WCSS (Within-Cluster Sum of Squares) and compute Silhouette scores for $K \in [2, 10]$ to determine and validate the optimal number of clusters.

In [ ]:
wcss = []
silhouette_scores = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    if k > 1:
        score = silhouette_score(X_scaled, kmeans.labels_)
        silhouette_scores.append(score)
        print(f"K = {k} | Silhouette Score: {score:.4f} | WCSS (Inertia): {kmeans.inertia_:.2f}")

In [ ]:
# Plot WCSS (Elbow Curve)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(list(k_range), wcss, marker='o', linestyle='--', color='#7C3AED')
plt.title('Elbow Method (WCSS vs K)', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.grid(True, linestyle=':', alpha=0.6)

# Plot Silhouette Scores
plt.subplot(1, 2, 2)
plt.plot(list(range(2, 11)), silhouette_scores, marker='s', linestyle='-', color='#06B6D4')
plt.title('Silhouette Score vs K', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### Result Interpretation
- **Elbow Method**: A clear elbow or inflection point is visible at **$K=5$**.
- **Silhouette Score**: The Silhouette Score reaches its peak value of **$0.5547$** at **$K=5$**.

Thus, the optimal number of clusters is mathematically validated to be **5**.

## 7. Train the K-Means Model & Assign Labels
We train the K-Means algorithm using the optimal number of clusters ($K=5$).

In [ ]:
optimal_k = 5
kmeans_model = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(X_scaled)

# Assign labels back to df
df['Cluster'] = cluster_labels
df.head()

## 8. Cluster Statistics
Let's check the mean values of each cluster to build our customer persona mappings.

In [ ]:
cluster_stats = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
print("=== Cluster Summary Statistics (Means) ===")
display(cluster_stats)

## 9. Customer Segments Visualization
Let's visualize the customer segments and their centroids in a 2D scatter plot, and plot the customer count per cluster.

In [ ]:
# Setup mapping and descriptive titles dynamically
cluster_means = df.groupby('Cluster')[features].mean()
cluster_mapping = {}
for cluster_id in range(optimal_k):
    inc = cluster_means.loc[cluster_id, 'Annual Income (k$)']
    spend = cluster_means.loc[cluster_id, 'Spending Score (1-100)']
    
    if inc > 70 and spend > 70:
        name = "Premium Customers"
        segment = "High Income, High Spending"
        rec = "Target with luxury promotions, exclusive loyalty rewards, and early access to new premium product lines."
    elif inc > 70 and spend < 40:
        name = "Cautious Customers"
        segment = "High Income, Low Spending"
        rec = "Offer high-quality value propositions, financial/investment rewards, and structured discount memberships."
    elif inc < 45 and spend > 60:
        name = "Spender Customers"
        segment = "Low Income, High Spending"
        rec = "Target with impulse-buy campaigns, flash sales, trendy marketing, and flexible payment options."
    elif inc < 45 and spend < 40:
        name = "Budget Customers"
        segment = "Low Income, Low Spending"
        rec = "Provide extreme-value discounts, bundle packages, and essential items marketing."
    else:
        name = "Standard Customers"
        segment = "Average Income, Average Spending"
        rec = "Maintain engagement with standard promotional newsletters and seasonal coupons."
        
    cluster_mapping[cluster_id] = {
        'Cluster Name': name,
        'Segment': segment,
        'Recommendation': rec
    }

df['Cluster Name'] = df['Cluster'].map(lambda c: cluster_mapping[c]['Cluster Name'])
df['Segment'] = df['Cluster'].map(lambda c: cluster_mapping[c]['Segment'])
df['Recommendation'] = df['Cluster'].map(lambda c: cluster_mapping[c]['Recommendation'])

In [ ]:
# Scatter Plot with Centroids
plt.figure(figsize=(10, 8))
colors = ['#7C3AED', '#06B6D4', '#22C55E', '#EF4444', '#F59E0B']

for i in range(optimal_k):
    cluster_df = df[df['Cluster'] == i]
    plt.scatter(
        cluster_df['Annual Income (k$)'], 
        cluster_df['Spending Score (1-100)'],
        s=85, 
        c=colors[i], 
        label=f"{cluster_mapping[i]['Cluster Name']} (Cluster {i})",
        edgecolors='black',
        linewidths=0.5,
        alpha=0.85
    )
    
# Get unscaled centroids
centroids_scaled = kmeans_model.cluster_centers_
centroids = scaler.inverse_transform(centroids_scaled)

plt.scatter(
    centroids[:, 0], 
    centroids[:, 1], 
    s=300, 
    c='white', 
    marker='*', 
    label='Centroids',
    edgecolors='black',
    linewidths=1.5
)

plt.title('Customer Segments and Centroids', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(loc='best', frameon=True, facecolor='white', edgecolor='gray')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Customer Count per Cluster plot
plt.figure(figsize=(9, 5))
sns.countplot(x='Cluster Name', data=df, hue='Cluster Name', palette=colors, legend=False)
plt.title('Customer Count per Cluster Segment', fontsize=14, fontweight='bold')
plt.xlabel('Customer Segment')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 10. Business Interpretation & Marketing Recommendations

Based on our segmentation, here is the profile and business strategy for each cluster:

In [ ]:
for cluster_id in sorted(cluster_mapping.keys()):
    mapping = cluster_mapping[cluster_id]
    stats = cluster_stats.loc[cluster_id]
    print(f"=== Cluster {cluster_id}: {mapping['Cluster Name']} ===")
    print(f"Description: {mapping['Segment']}")
    print(f"Mean Age: {stats['Age']:.1f} years | Mean Income: ${stats['Annual Income (k$)']:.1f}k | Mean Spending Score: {stats['Spending Score (1-100)']:.1f}")
    print(f"Recommendation: {mapping['Recommendation']}\n")

## 11. Save Models and Export Data
We serialize the trained K-Means model and standard scaler separately using joblib, and export the clustered customers dataframe to a CSV file.

In [ ]:
joblib.dump(kmeans_model, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
df.to_csv('clustered_customers.csv', index=False)
print("Successfully saved 'kmeans_model.pkl', 'scaler.pkl', and 'clustered_customers.csv'!")